In [7]:
import re
import torch
from collections import Counter

import torch
import torchvision.transforms as transforms
from torchvision.datasets import CocoCaptions
import matplotlib.pyplot as plt

In [8]:
image_dir = "./data/coco/train2017"
annotation_file = "./data/coco/annotations/captions_train2017.json"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

coco_dataset = CocoCaptions(
    root=image_dir,
    annFile=annotation_file,
    transform=transform
)

print("Number of samples:", len(coco_dataset))

loading annotations into memory...
Done (t=1.15s)
creating index...
index created!
Number of samples: 118287


In [10]:
def build_vocab(captions, min_freq=5):
    counter = Counter()

    for caption in captions:
        words = caption.split()
        counter.update(words)

    vocab = {
        "<pad>": 0,
        "<unk>": 1,
        "<start>": 2,
        "<end>": 3
    }

    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)

    return vocab


In [11]:
# ta bort tecken, göra alla bokstäver små
def clean_caption(caption):
    caption = caption.lower()
    caption = re.sub(r"[^a-z0-9\s]", "", caption)
    caption = re.sub(r"\s+", " ", caption).strip()
    return caption

#gör ett vocabulary av alla ord som förekommer minst 5 ggr d.v.s varje ord får ett nummer
def build_vocab(captions, min_freq=5):
    counter = Counter()

    for caption in captions:
        words = caption.split()
        counter.update(words)

    vocab = {
        "<pad>": 0,
        "<unk>": 1,
        "<start>": 2,
        "<end>": 3
    }

    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)

    return vocab

# gör om en caption till nummer-id 
def numericalize_caption(caption, vocab):
    caption = clean_caption(caption)

    tokens = ["<start>"]
    tokens += caption.split()
    tokens += ["<end>"]

    token_ids = []

    for token in tokens:
        token_id = vocab.get(token, vocab["<unk>"])
        token_ids.append(token_id)

    return token_ids

# paddar så att alla är samma längd
def pad_sequence(token_ids, max_length, pad_value=0):
    if len(token_ids) < max_length:
        token_ids = token_ids + [pad_value] * (max_length - len(token_ids))
    else:
        token_ids = token_ids[:max_length]

    return token_ids

# använder ovanstående funktioner för att göra hela captionen till tokens och en tensor
def tokenize_caption(caption, vocab, max_length=30):
    token_ids = numericalize_caption(caption, vocab)
    token_ids = pad_sequence(token_ids, max_length, pad_value=vocab["<pad>"])
    return torch.tensor(token_ids, dtype=torch.long)

In [12]:
#skapar en fil med preprocessad data
import os
import torch

def create_preprocessed_coco_file(
    coco_dataset,
    image_root,
    vocab,
    idx_to_word,
    save_path="./coco_preprocessed.pt",
    max_length=30
):
    samples = []

    for i in range(len(coco_dataset)):
        image_id = coco_dataset.ids[i]
        image_info = coco_dataset.coco.loadImgs(image_id)[0]
        filename = image_info["file_name"]
        image_path = os.path.join(image_root, filename)

        _, captions = coco_dataset[i]

        for caption in captions:
            token_ids = tokenize_caption(
                caption,
                vocab,
                max_length=max_length
            )

            samples.append({
                "image_path": image_path,
                "caption": caption,
                "token_ids": token_ids
            })

        if i % 1000 == 0:
            print(f"Processed {i}/{len(coco_dataset)} images")

    data = {
        "samples": samples,
        "vocab": vocab,
        "idx_to_word": idx_to_word,
        "max_length": max_length
    }

    torch.save(data, save_path)

    print(f"Saved preprocessed dataset to: {save_path}")
    print(f"Number of samples: {len(samples)}")

In [13]:
image_root = "./data/coco/train2017"

all_captions = []

for i in range(len(coco_dataset)):
    _, captions = coco_dataset[i]

    for caption in captions:
        all_captions.append(clean_caption(caption))

# skapa vocabulary
vocab = build_vocab(all_captions, min_freq=5)
idx_to_word = {idx: word for word, idx in vocab.items()}


create_preprocessed_coco_file(
    coco_dataset=coco_dataset,
    image_root=image_root,
    vocab=vocab,
    idx_to_word=idx_to_word,
    save_path="./coco_train_preprocessed.pt",
    max_length=30
)

Processed 0/118287 images
Processed 1000/118287 images
Processed 2000/118287 images
Processed 3000/118287 images
Processed 4000/118287 images
Processed 5000/118287 images
Processed 6000/118287 images
Processed 7000/118287 images
Processed 8000/118287 images
Processed 9000/118287 images
Processed 10000/118287 images
Processed 11000/118287 images
Processed 12000/118287 images
Processed 13000/118287 images
Processed 14000/118287 images
Processed 15000/118287 images
Processed 16000/118287 images
Processed 17000/118287 images
Processed 18000/118287 images
Processed 19000/118287 images
Processed 20000/118287 images
Processed 21000/118287 images
Processed 22000/118287 images
Processed 23000/118287 images
Processed 24000/118287 images
Processed 25000/118287 images
Processed 26000/118287 images
Processed 27000/118287 images
Processed 28000/118287 images
Processed 29000/118287 images
Processed 30000/118287 images
Processed 31000/118287 images
Processed 32000/118287 images
Processed 33000/118287 